# For colab environment

In [1]:
# !pip install nltk transformers==4.35.0 torch==2.6.0 torchvision==0.21.0 datasets accelerate==0.24.0 huggingface==0.0.1 datasets==2.14.7

In [1]:
import torch 
print(torch.cuda.is_available())
print(torch.__version__)

True
2.6.0+cu124


In [3]:
# !git clone https://github.com/BernardMoy/NLP-PCL-Classification.git

In [4]:
# %cd NLP-PCL-Classification/

In [2]:
!nvidia-smi

Mon Mar  2 15:15:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA TITAN Xp                Off |   00000000:02:00.0  On |                  N/A |
| 27%   39C    P8             19W /  250W |     427MiB /  12288MiB |     30%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Load train and validation data set

In [ ]:
import pandas as pd 
import numpy as np 
from matplotlib import pyplot as plt
from collections import Counter

df = pd.read_csv('../data/dontpatronizeme_pcl.tsv', sep='\t')

# Remove rows with NA labels 
df = df.dropna() 

# Add a bool_labels column for binary classification
df["bool_labels"] = df["label"] > 1   # is PCL if >1

# train val split 
train_labels = pd.read_csv('../data/train_semeval_parids-labels.csv')["par_id"]
val_labels = pd.read_csv('../data/dev_semeval_parids-labels.csv')["par_id"]
df_train = df[df["par_id"].isin(train_labels)].reset_index() 
df_val = df[df["par_id"].isin(val_labels)].reset_index() 


# Text Cleaning

In [4]:
# Remove special characters
SPECIAL_CHARACTERS = ['&amp;', '&lt;', '&gt;', '<h>', '\n', '\t']
for char in SPECIAL_CHARACTERS:
    df_train["text"] = df_train["text"].str.replace(char, "")
    df_val["text"] = df_val["text"].str.replace(char, "")


print(df_train["text"].iloc[55])


# Replace numbers with 0
df_train["text"] = df_train["text"].str.replace(r"\d+", "0", regex=True)
df_val["text"] = df_val["text"].str.replace(r"\d+", "0", regex=True)

print(df_train["text"].iloc[3])

People who are homeless , those who were once homeless , those working with the homeless and concerned New Zealanders are being asked to share their experiences and solutions to this growing issue with the Cross-Party Homelessness Inquiry . More
Council customers only signs would be displayed . Two of the spaces would be reserved for disabled persons and there would be five P0 spaces and eight P0 ones .


# Oversample the minority class
For each keyword category, inflate the number of positive examples to a certain percentage

In [5]:
POSITIVE_PERCENTAGE = 25


# Find all the unique keywords in the training dataset
keywords = pd.unique(df_train["keyword"])


# Extract the sub-dataset for each keyword
for keyword in keywords:
    subdata = df_train[df_train["keyword"] == keyword]
    rows = subdata.shape[0]


    # Find the number of positive entires x
    subdata_positive = subdata[subdata["bool_labels"] == True]
    positive_rows = subdata_positive.shape[0]


    # Calculate the number of additional samples needed to make the positive class reach the desired percentage
    # (p+x)/(r+x) = POS PERCENTAGE
    n_samples = round((100*positive_rows-POSITIVE_PERCENTAGE*rows)/(POSITIVE_PERCENTAGE-100)*1.0)


    # Sample with replacement from the sub dataset and add new rows
    sampled = subdata_positive.sample(n_samples, replace=True).reset_index(drop=True)
   
    # concat with the main df
    df_train = pd.concat([df_train, sampled], ignore_index=True)


df_train["bool_labels"].value_counts()


bool_labels
False    7581
True     2527
Name: count, dtype: int64

# Tokenization

In [6]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification, AutoConfig, Trainer, TrainingArguments

tokenizer = RobertaTokenizer.from_pretrained("roberta-base") 

# # define special tokens for separating text 
# special_tokens = {"additional_special_tokens": ["<SEP>"]}
# tokenizer.add_special_tokens(special_tokens) 

# Create text with contextual information 
def tokenize(df): 
    text_with_context = df["text"]

    encoding = tokenizer(
        text_with_context.tolist(), 
        padding="max_length",   # Add padding to shorter sentences 
        max_length=256,
        truncation = True, 
        return_attention_mask = True 
    )

    return encoding

/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


# Convert to pyTorch dataset

In [7]:
import torch 
from torch.utils.data import DataLoader, TensorDataset
from datasets import Dataset

def to_dataset(df): 
    # Obtain tokens (input_ids, attention_mask) from the dataset 
    encoding = tokenize(df) 

    # Return huggingface dataset 
    return Dataset.from_dict({
        "input_ids": encoding["input_ids"], 
        "attention_mask": encoding["attention_mask"], 
        "label": df["bool_labels"].values 
    })

In [8]:
train_dataset = to_dataset(df_train)
val_dataset = to_dataset(df_val) 

# set to torch format 
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Training 

In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(pred):
    logits, labels = pred
    predictions = np.argmax(logits, axis=-1)

    # Calculate metrics 
    accuracy = accuracy_score(labels, predictions) 
    precision = precision_score(labels, predictions) 
    recall = recall_score(labels, predictions) 
    f1 = f1_score(labels, predictions) 

    return {
        "accuracy": accuracy, 
        "precision": precision, 
        "recall": recall, 
        "f1": f1 
    }


In [14]:
# Load roberta sequence classification model 
config = AutoConfig.from_pretrained("roberta-base", num_labels=2)  # Binary classification
model = RobertaForSequenceClassification.from_pretrained("roberta-base", config = config)
# model.resize_token_embeddings(len(tokenizer)) 

# Core hyperparameters 
BATCH_SIZE = 32
N_EPOCHS = 5 

# Set up training arguments 
training_args = TrainingArguments(
    fp16=True, 
    num_train_epochs=N_EPOCHS, 
    learning_rate=2e-5, 
    weight_decay=0.01,
    warmup_steps=500, 
    save_strategy="epoch",  # low disk space 
    load_best_model_at_end=True, 
    metric_for_best_model='f1',
    logging_steps=50,
    output_dir="./checkpoints/only_oversample", 
    evaluation_strategy="epoch", 
    per_device_eval_batch_size=BATCH_SIZE, 
    per_device_train_batch_size=BATCH_SIZE, 
)

# Set up trainer 
trainer = Trainer(
    model = model, 
    args = training_args, 
    train_dataset=train_dataset, 
    eval_dataset=val_dataset, 
    compute_metrics=compute_metrics
)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.weight', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/vol/bitbucket/bm1325/dl_cw_1/dlvenv/lib/python3.12/site-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [15]:
trainer.train() 

  0%|          | 0/1580 [00:00<?, ?it/s]

{'loss': 0.6208, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.16}
{'loss': 0.5678, 'learning_rate': 4.000000000000001e-06, 'epoch': 0.32}
{'loss': 0.5255, 'learning_rate': 5.9600000000000005e-06, 'epoch': 0.47}
{'loss': 0.4208, 'learning_rate': 7.960000000000002e-06, 'epoch': 0.63}
{'loss': 0.3655, 'learning_rate': 9.960000000000001e-06, 'epoch': 0.79}
{'loss': 0.3471, 'learning_rate': 1.1920000000000001e-05, 'epoch': 0.95}


  0%|          | 0/66 [00:00<?, ?it/s]

{'eval_loss': 0.2203952968120575, 'eval_accuracy': 0.9106545628284759, 'eval_precision': 0.524, 'eval_recall': 0.6582914572864321, 'eval_f1': 0.5835189309576837, 'eval_runtime': 17.1304, 'eval_samples_per_second': 122.18, 'eval_steps_per_second': 3.853, 'epoch': 1.0}
{'loss': 0.2617, 'learning_rate': 1.392e-05, 'epoch': 1.11}
{'loss': 0.2337, 'learning_rate': 1.5920000000000003e-05, 'epoch': 1.27}
{'loss': 0.2257, 'learning_rate': 1.792e-05, 'epoch': 1.42}
{'loss': 0.2005, 'learning_rate': 1.9920000000000002e-05, 'epoch': 1.58}
{'loss': 0.2089, 'learning_rate': 1.9111111111111113e-05, 'epoch': 1.74}
{'loss': 0.1912, 'learning_rate': 1.8185185185185186e-05, 'epoch': 1.9}


  0%|          | 0/66 [00:00<?, ?it/s]

...

  0%|          | 0/66 [00:00<?, ?it/s]

{'eval_loss': 0.41441038250923157, 'eval_accuracy': 0.9173435260391782, 'eval_precision': 0.5590909090909091, 'eval_recall': 0.6180904522613065, 'eval_f1': 0.5871121718377088, 'eval_runtime': 17.1098, 'eval_samples_per_second': 122.327, 'eval_steps_per_second': 3.857, 'epoch': 4.0}
{'loss': 0.0237, 'learning_rate': 5.240740740740741e-06, 'epoch': 4.11}
{'loss': 0.0239, 'learning_rate': 4.314814814814815e-06, 'epoch': 4.27}
{'loss': 0.0212, 'learning_rate': 3.3888888888888893e-06, 'epoch': 4.43}
{'loss': 0.0117, 'learning_rate': 2.462962962962963e-06, 'epoch': 4.59}
{'loss': 0.0203, 'learning_rate': 1.5370370370370372e-06, 'epoch': 4.75}
{'loss': 0.0094, 'learning_rate': 6.111111111111112e-07, 'epoch': 4.91}


  0%|          | 0/66 [00:00<?, ?it/s]

{'eval_loss': 0.45513710379600525, 'eval_accuracy': 0.9254658385093167, 'eval_precision': 0.6257309941520468, 'eval_recall': 0.5376884422110553, 'eval_f1': 0.5783783783783784, 'eval_runtime': 17.3541, 'eval_samples_per_second': 120.605, 'eval_steps_per_second': 3.803, 'epoch': 5.0}
{'train_runtime': 1394.6941, 'train_samples_per_second': 36.237, 'train_steps_per_second': 1.133, 'train_loss': 0.16891603124669835, 'epoch': 5.0}


TrainOutput(global_step=1580, training_loss=0.16891603124669835, metrics={'train_runtime': 1394.6941, 'train_samples_per_second': 36.237, 'train_steps_per_second': 1.133, 'train_loss': 0.16891603124669835, 'epoch': 5.0})

In [16]:
trainer.evaluate()

  0%|          | 0/66 [00:00<?, ?it/s]

{'eval_loss': 0.41441038250923157,
 'eval_accuracy': 0.9173435260391782,
 'eval_precision': 0.5590909090909091,
 'eval_recall': 0.6180904522613065,
 'eval_f1': 0.5871121718377088,
 'eval_runtime': 16.6045,
 'eval_samples_per_second': 126.05,
 'eval_steps_per_second': 3.975,
 'epoch': 5.0}

# Save model

In [ ]:
trainer.save_model('../models/model_only_oversample')

# Write results

In [ ]:
import os 
# must pass abs path here 
only_oversample_model = RobertaForSequenceClassification.from_pretrained(os.path.abspath("/vol/bitbucket/bm1325/NLP-PCL-Classification/models/model_only_oversample"), local_files_only = True)
trainer = Trainer(model = only_oversample_model)

model_pred = trainer.predict(val_dataset)

# extract the class with higher logit value 
y_pred = model_pred.predictions.argmax(axis=1) 
y_true = df_val["bool_labels"]

with open("../evaluation/only_oversample.txt", "w") as f: 
    f.write('\n'.join(str(x) for x in y_pred)) 


  0%|          | 0/262 [00:00<?, ?it/s]